# InsightForge — AI-Powered Business Intelligence Assistant
**Advanced Generative AI Capstone**

InsightForge helps small and mid-size businesses turn raw sales data into insights and recommendations, using **pandas**, **LangChain**, **Retrieval-Augmented Generation (RAG)** and an **LLM**.

| Part | Step | Where in this notebook |
|---|---|---|
| 1 | 1. Data preparation | Section 1 |
| 1 | 2. Knowledge base creation (sales data + research PDFs) | Section 2 |
| 1 | 3. LLM application development (advanced summary, custom retriever, prompt engineering) | Section 3 |
| 1 | 4. Chain prompts | Section 4 |
| 1 | 5. RAG system setup | Section 5 |
| 1 | 6. Memory integration | Section 6 |
| 2 | 7. Model evaluation (QAEvalChain), data visualization, Streamlit UI | Section 7 |

The reusable code lives in the `insightforge/` package; this notebook runs and explains each step.

In [ ]:
# !pip install -r requirements.txt
import os, getpass
import pandas as pd
from IPython.display import display

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

## 1. Data preparation
Per the brief, the data is **pre-prepared**, so the focus is analysis rather than cleaning. `load_data` validates the expected columns and adds derived fields used for analysis: `Year`, `Quarter`, `Month`, and `Age_Group`.

In [ ]:
from insightforge.data_analysis import load_data, explore, full_years

df = load_data("data/sales_data.csv")
print("Complete years:", full_years(df))
df.head()

## 2. Knowledge base creation
### 2a. Load and explore the dataset

In [ ]:
info = explore(df)
print(f"Rows: {info['rows']:,}  |  Date range: {info['date_range'][0]} to {info['date_range'][1]}")
print("Products:", info["products"]); print("Regions:", info["regions"])
df.info()

### 2b. Organise the data into a structured format for retrieval
Each computed summary becomes one knowledge-base entry, tagged with a **topic** (`time`, `product`, `region`, `customer`, `statistics`, `satisfaction`, …). These tags let the custom retriever route a question to the right statistics, and the same entries are embedded into a FAISS vector store for semantic search.

In [ ]:
from insightforge.knowledge_base import build_knowledge_base

kb = build_knowledge_base(df)
print(f"{len(kb)} knowledge-base entries\n")
for e in kb:
    print(f"[{e.topic:<14}] {e.title}")

In [ ]:
print(kb[0].text)   # the overview entry

### 2c. Research documents (PDF folder)
The course dataset includes four business-intelligence research papers (BI approaches, AI-driven business model innovation, Walmart sales analysis, and time-series prediction). They are loaded with `pypdf`, split into overlapping ~1,200-character chunks with source and page metadata, and embedded into a second FAISS index. The retriever adds the most relevant passages to strategy questions ("how can we…", "best practice", "AI", "forecast"), so recommendations can cite published research while every number still comes from the sales data.

In [ ]:
from collections import Counter
from insightforge.knowledge_base import load_pdf_chunks, keyword_search

research_chunks = load_pdf_chunks("data/pdfs")
print(f"{len(research_chunks)} research chunks"); print(Counter(c.metadata["source"] for c in research_chunks))
for c in keyword_search(research_chunks, "How can AI improve our business model?", k=2):
    print("\n•", c.title, "\n ", c.text[:300], "…")

## 3. LLM application development
### 3a. Advanced data summary
Key metrics and trends, computed with pandas:
1. Sales performance by time period  2. Product and regional analysis  3. Customer segmentation by demographics  4. Statistical measures

In [ ]:
from insightforge.data_analysis import full_summary
s = full_summary(df)
s["kpis"]

In [ ]:
print("1. Sales performance by year"); display(s["yearly"])
print("   ...and by quarter"); display(s["quarterly"])

In [ ]:
print("2. Product analysis"); display(s["products"])
print("   Regional analysis"); display(s["regions"])
print("   Product x Region"); display(s["product_region"])

In [ ]:
print("3. Customer segmentation by gender"); display(s["gender"])
print("   ...by age group"); display(s["age_groups"])
print("   Best-selling product per age group"); display(s["top_product_per_age"])

In [ ]:
print("4. Statistical measures"); display(s["statistics"])
print("   Correlations"); display(s["correlations"])

### 3b. Integration with the RAG system — custom retriever
`InsightRetriever` (in `insightforge/rag.py`) subclasses LangChain's `BaseRetriever` and combines three retrieval paths:

1. **Entity filtering with pandas** — products, regions, years or genders named in a question are computed live (e.g. *Widget A in the North in 2023*).
2. **Topic routing** — keywords map the question to the relevant pre-computed summary tables.
3. **Semantic search** — FAISS similarity over the knowledge base catches loosely worded questions.

This hybrid design means the LLM is always handed exact, computed numbers rather than asked to do arithmetic over raw rows.

In [ ]:
from insightforge.knowledge_base import route_query, retrieve

for q in ["How did Widget A do in the North in 2023?",
          "What's the median and standard deviation of sales?",
          "Which age group buys the most?"]:
    print(f"Q: {q}\n  topics -> {route_query(q)}")
    for e in retrieve(df, kb, q):
        print("   •", e.title)
    print()

In [ ]:
print(retrieve(df, kb, "How did Widget A do in the North in 2023?")[0].text)

### 3c. Prompt engineering
The prompts (in `insightforge/rag.py`) use these techniques to keep answers accurate:
- **Role prompting** — the model acts as a senior BI analyst writing for a business owner.
- **Grounding rule** — *only use numbers that appear in the provided data context; never invent figures*, and say what's missing when the data can't answer.
- **Task decomposition** — fact extraction is separated from interpretation (see Section 4).
- **Output structure** — every answer has *Answer / Insights / Recommendations* sections and a length limit.

In [ ]:
from insightforge.rag import CONDENSE_PROMPT, ANALYSIS_PROMPT, INSIGHT_PROMPT
for name, p in [("CONDENSE", CONDENSE_PROMPT), ("ANALYSIS", ANALYSIS_PROMPT), ("INSIGHT", INSIGHT_PROMPT)]:
    print(f"===== {name} =====")
    print(p.pretty_repr(), "\n")

## 4. Chain prompts
Each question flows through **three chained prompts** built with LangChain Expression Language (`prompt | llm | parser`):

1. **Condense** — uses chat history to rewrite a follow-up ("what about the West?") as a standalone question.
2. **Analysis** — reads the retrieved statistics and extracts only the relevant facts, with exact numbers.
3. **Insight** — turns those verified facts into an answer, insights and recommendations.

Splitting extraction from interpretation keeps the final answer coherent and stops the model from blending numbers across tables.

## 5. RAG system setup
Building the assistant creates the knowledge base, embeds it into FAISS, wires the custom retriever, and assembles the chains.

In [ ]:
from insightforge.rag import InsightForgeAssistant

assistant = InsightForgeAssistant(df, model="gpt-4o-mini", use_vectorstore=True,
                                 pdf_folder="data/pdfs")
res = assistant.ask("Which product is our best performer, and what should we do about the weakest one?")
print("Sources used:", res["sources"], "\n")
print("--- Step 1: extracted facts ---\n", res["facts"], "\n")
print("--- Step 2: final answer ---\n", res["answer"])

In [ ]:
# A strategy question: numbers come from the data, supporting ideas from the research PDFs
res = assistant.ask("How can we use AI to grow sales in our weakest region? Use the research.")
print("Sources used:", res["sources"], "\n"); print(res["answer"])

## 6. Memory integration
The assistant stores each exchange in a `InMemoryChatMessageHistory` and passes the most recent exchanges (a window of 6) into the condense and insight prompts. Follow-up questions can therefore refer back to earlier ones.

In [ ]:
r1 = assistant.ask("Which region had the highest total sales?")
print("Q1:", r1["answer"], "\n")

r2 = assistant.ask("And what's the best-selling product there?")   # 'there' must be resolved from memory
print("Rewritten with memory ->", r2["standalone_question"])
print("Q2:", r2["answer"])

In [ ]:
print(f"{len(assistant.memory.messages)} messages in memory")

## 7. External tool integration (LLMOps)
### 7a. Model evaluation with QAEvalChain
Reference answers are computed directly with pandas, so the evaluation checks the assistant's answers against the true numbers. Each question is answered **without memory** so tests stay independent, then `QAEvalChain` grades each answer CORRECT or INCORRECT.

In [ ]:
from insightforge.evaluation import build_test_set, evaluate_assistant, summarize_eval

pd.DataFrame(build_test_set(df))

In [ ]:
results = evaluate_assistant(assistant, assistant.llm)
print(summarize_eval(results))
results[["Question", "Expected", "Grade", "Latency_s"]]

In [ ]:
os.makedirs("outputs", exist_ok=True)
results.to_csv("outputs/evaluation_results.csv", index=False)

### 7b. Data visualization

In [ ]:
%matplotlib inline
from insightforge import visualizations as viz
viz.sales_trend(df)

In [ ]:
viz.product_performance(df)

In [ ]:
viz.regional_analysis(df)

In [ ]:
viz.customer_segmentation(df)

In [ ]:
viz.satisfaction_vs_sales(df)

In [ ]:
viz.save_all(df, "outputs/charts")   # PNGs for the report

### 7c. Streamlit UI
Run from the project folder:
```bash
streamlit run app.py
```
The app has four tabs:
- **Ask InsightForge** — chat with memory, clickable example questions, sources for every answer, and a "How this answer was built" panel showing the rewritten question, extracted facts and retrieved context.
- **Dashboard** — KPI cards plus all charts, filterable by product and region.
- **Data & Stats** — summary tables and the raw data.
- **Evaluation** — runs the QAEvalChain test suite and shows accuracy, per-question grades and latency.

Users can upload their own CSV with the same columns.

## Conclusion
InsightForge meets each objective in the brief:
- **Analyze business data** — pandas summaries of time, product, region, demographics and statistical measures.
- **Generate insights and recommendations** — a hybrid custom retriever plus three chained prompts with memory, grounded in computed numbers.
- **Visualize data insights** — four required chart groups plus satisfaction analysis, in the notebook and the Streamlit dashboard.
- **LLMOps** — QAEvalChain evaluation against pandas ground truth, with latency tracking.

**Possible next steps:** add a pandas-agent fallback for questions outside the knowledge base, persist memory per user, and log evaluations over time to monitor drift.